# DACON ETRI post-ID339 candidate parity runner

Report-only Kaggle runner for ID337/ID339 candidate parity. It recomputes the two generated-unsubmitted candidates on Kaggle CPU and compares the rebuilt OOF/test predictions with the immutable base prediction artifacts packaged in the project dataset. It creates only small diagnostic JSON/MD files; it does **not** create DACON submissions, promoted artifacts, or submission-log rows.

## 1. Locate the packaged project payload

The runner accepts either a `source.zip` from the Kaggle dataset package or an unpacked project tree under `/kaggle/input`. It then works from an isolated temporary project directory and stages only parity diagnostics under `/kaggle/working/kaggle_outputs`.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path


def run(cmd: str) -> None:
    print(f"\n$ {cmd}", flush=True)
    env = os.environ.copy()
    env.setdefault("PYTHONHASHSEED", "0")
    env.setdefault("OMP_NUM_THREADS", "2")
    env.setdefault("OPENBLAS_NUM_THREADS", "2")
    env.setdefault("MKL_NUM_THREADS", "2")
    env.setdefault("NUMEXPR_NUM_THREADS", "2")
    subprocess.run(cmd, shell=True, check=True, env=env)


is_kaggle = Path("/kaggle/input").exists()
work_dir = Path("/kaggle/working") if is_kaggle else Path.cwd()
project_root = Path("/tmp/dacon_project_candidate_parity_post_id339") if is_kaggle else Path.cwd()
output_root = work_dir / "kaggle_outputs" if is_kaggle else Path("kaggle_outputs")
output_root.mkdir(parents=True, exist_ok=True)
os.environ["KAGGLE_OUTPUT_DIR"] = str(output_root)

for key, value in {
    "PYTHONHASHSEED": "0",
    "OMP_NUM_THREADS": "2",
    "OPENBLAS_NUM_THREADS": "2",
    "MKL_NUM_THREADS": "2",
    "NUMEXPR_NUM_THREADS": "2",
}.items():
    os.environ.setdefault(key, value)

required_payload_paths = [
    Path("scripts/diagnose_candidate_parity_post_id339.py"),
    Path("scripts/promote_id178_s1_seed45502_candidate.py"),
    Path("scripts/promote_id178_q123_seqsum_s080_id337_s1_candidate.py"),
    Path("artifacts/base_predictions/base_oof_id_178_id173_fine_highc_random_conv_sequence_overlay_cv0.538638.csv"),
    Path("artifacts/base_predictions/base_test_id_178_id173_fine_highc_random_conv_sequence_overlay_cv0.538638.csv"),
    Path("artifacts/base_predictions/base_oof_id_337_id178_s1_seed45502_randomconv_w005_safety_cv0.538066.csv"),
    Path("artifacts/base_predictions/base_test_id_337_id178_s1_seed45502_randomconv_w005_safety_cv0.538066.csv"),
    Path("artifacts/base_predictions/base_oof_id_339_id178_q123_seqsum_s080_id337_s1_safety_cv0.536441.csv"),
    Path("artifacts/base_predictions/base_test_id_339_id178_q123_seqsum_s080_id337_s1_safety_cv0.536441.csv"),
    Path("submissions/submission_log.csv"),
]


def payload_score(root: Path) -> int:
    return sum((root / rel).exists() for rel in required_payload_paths)


if is_kaggle:
    print("Detected Kaggle environment; locating project payload...")
    source_zips = sorted(Path("/kaggle/input").glob("**/source.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
    project_root.mkdir(parents=True, exist_ok=True)
    if source_zips:
        print("Using source.zip:", source_zips[0])
        with zipfile.ZipFile(source_zips[0]) as zf:
            zf.extractall(project_root)
    else:
        candidates = sorted(Path("/kaggle/input").glob("**/scripts/diagnose_candidate_parity_post_id339.py"))
        if not candidates:
            raise FileNotFoundError("Could not find source.zip or candidate parity script under /kaggle/input")
        source_root = candidates[0].parents[1]
        print("Using unpacked project source root:", source_root)
        if project_root.exists():
            shutil.rmtree(project_root)
        shutil.copytree(source_root, project_root)
        print("Copied unpacked project to writable root:", project_root)
    os.chdir(project_root)
else:
    print("Running outside Kaggle; using current workspace:", project_root)

missing_payload = [str(rel) for rel in required_payload_paths if not (Path.cwd() / rel).exists()]
print("payload score =", payload_score(Path.cwd()), "/", len(required_payload_paths))
if missing_payload:
    raise FileNotFoundError("Missing required payload files: " + json.dumps(missing_payload, indent=2))
print("Project root:", Path.cwd())

The payload check is intentionally strict: if ID337/ID339 base artifacts are absent, the notebook fails before doing any compute. That protects the parity report from silently comparing against the wrong files.

## 2. Ensure runtime dependencies only if missing

Kaggle usually has these packages already. The cell checks imports first and installs only what is absent.

In [ ]:
if is_kaggle:
    import importlib.util

    required = {
        "nbformat": "nbformat",
        "numpy": "numpy",
        "pandas": "pandas",
        "pyarrow": "pyarrow",
        "fastparquet": "fastparquet",
        "sklearn": "scikit-learn",
    }
    missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
    print("missing packages =", missing)
    if missing:
        run(f"{sys.executable} -m pip install -q {' '.join(missing)}")
else:
    print("Skipping dependency installation outside Kaggle.")

No new project dependency is introduced here; this is a runtime-only Kaggle safeguard for a report notebook.

## 3. Rebuild processed caches if needed

The parity script uses the same project helpers as the promoted candidates. If processed day-level caches are absent in the packaged dataset, this cell rebuilds them from raw parquet before the diagnostic runs.

In [ ]:
from pathlib import Path

import pandas as pd

train_cache = Path("data/processed/df_train_raw.parquet")
test_cache = Path("data/processed/df_test_raw.parquet")
if train_cache.exists() and test_cache.exists():
    print("Processed caches already exist; skipping rebuild.")
else:
    from src.data.make_dataset import build_training_dataset

    raw_dir = Path("data/raw")
    sensor_dir = raw_dir / "ch2025_data_items"
    if not sensor_dir.exists():
        raise FileNotFoundError(f"Missing raw sensor dir: {sensor_dir}")
    sensor_data = {}
    for parquet_path in sorted(sensor_dir.glob("*.parquet")):
        print("Loading", parquet_path)
        sensor_data[parquet_path.stem] = pd.read_parquet(parquet_path)
    metrics = pd.read_csv(raw_dir / "ch2026_metrics_train.csv")
    sample = pd.read_csv(raw_dir / "ch2026_submission_sample.csv")
    train_df = build_training_dataset(metrics, sensor_data)
    test_df = build_training_dataset(sample, sensor_data)
    train_cache.parent.mkdir(parents=True, exist_ok=True)
    train_df.to_parquet(train_cache, index=False)
    test_df.to_parquet(test_cache, index=False)
    print("Wrote processed caches:", train_df.shape, test_df.shape)

The cache step is skipped when the source dataset already includes processed parquet files, which keeps the parity run small.

## 4. Run ID337/ID339 parity diagnostic

This cell recomputes the two candidate constructions and compares rebuilt predictions against the packaged immutable base predictions. It is the core parity gate.

In [ ]:
import time
from pathlib import Path

RUN_STARTED_TS = time.time()
PARITY_OUTPUT = Path("reports/candidate_parity_post_id339_20260618.json")
run(f"{sys.executable} scripts/diagnose_candidate_parity_post_id339.py --output {PARITY_OUTPUT}")

A passing report means Kaggle CPU can reproduce the local ID337/ID339 candidate artifacts within the strict review thresholds. It still does not authorize a DACON submission.

## 5. Stage only parity outputs

Only the parity JSON/MD and a small manifest are copied to Kaggle outputs. Submission files, base predictions, notebooks/history files, and submission logs are deliberately not staged.

In [ ]:
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

out_dir = Path(os.environ.get("KAGGLE_OUTPUT_DIR", "kaggle_outputs"))
out_dir.mkdir(parents=True, exist_ok=True)
patterns = [
    "reports/candidate_parity_post_id339_20260618.json",
    "reports/candidate_parity_post_id339_20260618.md",
]
copied = []
for pattern in patterns:
    for path in sorted(Path(".").glob(pattern)):
        if not path.is_file():
            continue
        dest = out_dir / path.name
        shutil.copy2(path, dest)
        copied.append(str(path))
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "command": "post-ID339 ID337/ID339 candidate parity report-only runner",
    "copied_artifacts": copied,
    "note": "Report-only parity runner. No DACON submission, submission-log row, base prediction artifact, or promoted notebook is created here.",
}
manifest_path = out_dir / "remote_candidate_parity_post_id339_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(manifest, indent=2, ensure_ascii=False))

### Data Analysis Key Findings
- This notebook is a parity gate, not a leaderboard experiment.
- The only expected outputs are `candidate_parity_post_id339_20260618.json`, `candidate_parity_post_id339_20260618.md`, and `remote_candidate_parity_post_id339_manifest.json`.

### Insights or Next Steps
- Pull the Kaggle outputs locally and compare the parity status before considering any user-approved public probe.
- If parity fails, do not submit ID337 or ID339; investigate environment/package differences first.